# CineInfini — Unit tests

Run the entire pytest suite from a notebook. Useful for CI and for
sanity-checking after a fresh install or weight update.

This notebook is **read-only** — it doesn't modify anything outside
`/tmp/cineinfini_test/`. Re-run it as many times as you want.


## 1. Verify the install

In [ ]:
import sys, subprocess, os
print("Python:", sys.version.split()[0])
result = subprocess.run([sys.executable, "-c",
    "import cineinfini; print('cineinfini', cineinfini.__version__)"],
    capture_output=True, text=True, cwd=os.environ.get('CINEINFINI_REPO', '.'))
print(result.stdout)
print(result.stderr)


## 2. Run the fast unit tests (excludes integration & slow)

In [ ]:
import subprocess, os
repo = os.environ.get('CINEINFINI_REPO', '.')
env = {**os.environ, 'PYTHONPATH': f'{repo}/src'}
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v',
     '-m', 'not integration and not slow', '--tb=short'],
    cwd=repo, env=env, capture_output=True, text=True, timeout=300,
)
print(result.stdout[-4000:])
if result.returncode:
    print("STDERR:", result.stderr[-1000:])
print(f"\nExit code: {result.returncode}")


## 3. Targeted: just the v0.4.8.x deltas

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/test_orchestrator_registry.py',
     'tests/test_datasets.py',
     'tests/test_partial_zip.py',
     'tests/test_optional_models.py',
     'tests/test_competitive_parity.py',
     'tests/test_profiles.py',
     '-v', '--tb=line'],
    cwd=repo, env=env, capture_output=True, text=True, timeout=180,
)
print(result.stdout[-3500:])
print(f"\nExit code: {result.returncode}")


## 4. Test counts by file (sanity check)

In [ ]:
import subprocess, re
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '--collect-only', '-q'],
    cwd=repo, env=env, capture_output=True, text=True,
)
out = result.stdout
counts = {}
for line in out.splitlines():
    m = re.match(r'tests/(\w+)\.py::', line)
    if m:
        counts[m.group(1)] = counts.get(m.group(1), 0) + 1
for name, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {name:35s} {n:3d} tests")
print(f"\nTotal: {sum(counts.values())} tests across {len(counts)} files")


## What this notebook validates

If all three cells above show green:

- the package imports cleanly
- every unit + module + registry + config test passes
- the v0.4.8.x deltas (registry, datasets, partial-ZIP, FAST-VQA pinning,
  competitive-parity exporters, profiles) are all green

If a cell fails, the output above tells you which test failed and why.
